# Notebook 06 — Report and Comparison

Goal: merge automatic metrics, LLM-judge results, and manual audit notes into
the final comparison tables that answer the four assignment questions.

> **Pre-requisites**: Notebooks 04 and 05 must have run.
> The filled-in `safety_audit.csv` files must be uploaded back to
> `outputs/trackA/` and `outputs/trackB/` before running Section 4.

## 1. Setup


In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/abhishek1998s/medical-reasoning-llm.git'
REPO_DIR = '/kaggle/working/medical-reasoning-llm'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 2. Load All Outputs


In [ ]:
import json, pandas as pd
from pathlib import Path

metrics = json.loads(Path('outputs/metrics_summary.json').read_text(encoding='utf-8'))
pred_a  = pd.read_csv('outputs/trackA/predictions.csv')
pred_b  = pd.read_csv('outputs/trackB/predictions.csv')

# Judge and audit files are optional — notebooks are designed to run even
# if judge / audit haven't been completed yet.
def _load_csv(path):
    p = Path(path)
    if p.exists() and p.stat().st_size > 0:
        return pd.read_csv(p)
    print(f'  [missing] {path}')
    return pd.DataFrame()

judge_a = _load_csv('outputs/trackA/judged.csv')
judge_b = _load_csv('outputs/trackB/judged.csv')
audit_a = _load_csv('outputs/trackA/safety_audit.csv')
audit_b = _load_csv('outputs/trackB/safety_audit.csv')

print(f'pred_a: {len(pred_a)} rows   pred_b: {len(pred_b)} rows')
print(f'judge_a: {len(judge_a)} rows  judge_b: {len(judge_b)} rows')
print(f'audit_a: {len(audit_a)} rows  audit_b: {len(audit_b)} rows')

## 3. Core Comparison Table


In [ ]:
rows = []
for track, data in metrics.items():
    rows.append({
        'track':                 track,
        'exact_match':           data.get('exact_match'),
        'rouge_l':               data.get('rouge_l'),
        'mean_output_tokens':    data.get('mean_output_tokens'),
        'mean_generation_time_s':data.get('mean_generation_time_s'),
        'mean_tokens_per_sec':   data.get('mean_tokens_per_sec'),
        'n':                     data.get('n'),
    })
comparison = pd.DataFrame(rows).set_index('track')
comparison.to_csv('outputs/final_comparison_table.csv')
print('Saved outputs/final_comparison_table.csv')
comparison

In [ ]:
# Per-sample merged table: one row per question, Track A and B side by side
merged = pred_a[['sample_id','question','reference','prediction','output_tokens',
                  'generation_time_s','finish_reason','truncated']].copy()
merged = merged.rename(columns={
    'prediction':'pred_A', 'output_tokens':'tokens_A',
    'generation_time_s':'time_A', 'finish_reason':'finish_A', 'truncated':'trunc_A'
})
b_cols = pred_b[['sample_id','prediction','output_tokens','generation_time_s',
                  'finish_reason','truncated']].rename(columns={
    'prediction':'pred_B', 'output_tokens':'tokens_B',
    'generation_time_s':'time_B', 'finish_reason':'finish_B', 'truncated':'trunc_B'
})
merged = merged.merge(b_cols, on='sample_id', how='inner')

# Add per-sample EM and ROUGE-L deltas
from src.data_formatting import extract_answer_for_scoring
from src.metrics import exact_match, rouge_l_score
merged['em_A']    = [exact_match(extract_answer_for_scoring(p,'A'), r)
                     for p, r in zip(merged['pred_A'], merged['reference'])]
merged['em_B']    = [exact_match(extract_answer_for_scoring(p,'B'), r)
                     for p, r in zip(merged['pred_B'], merged['reference'])]
merged['rouge_A'] = [rouge_l_score(extract_answer_for_scoring(p,'A'), r)
                     for p, r in zip(merged['pred_A'], merged['reference'])]
merged['rouge_B'] = [rouge_l_score(extract_answer_for_scoring(p,'B'), r)
                     for p, r in zip(merged['pred_B'], merged['reference'])]
merged['delta_em']    = merged['em_A']    - merged['em_B']
merged['delta_rouge'] = merged['rouge_A'] - merged['rouge_B']
merged['delta_tokens']= merged['tokens_A'] - merged['tokens_B']
merged['delta_time']  = merged['time_A']   - merged['time_B']

merged.to_csv('outputs/per_sample_comparison.csv', index=False)
print(f'Saved outputs/per_sample_comparison.csv  ({len(merged)} rows)')
merged[['sample_id','em_A','em_B','delta_em','delta_rouge','delta_tokens','delta_time','trunc_A','trunc_B']].head(20)

## 4. Judge and Safety Summaries


In [ ]:
def judge_summary(df):
    if df.empty:
        return {'note': 'no judge results yet'}
    numeric_cols = df.select_dtypes('number').columns.tolist()
    return {c: round(float(df[c].mean()), 3) for c in numeric_cols}

def audit_summary(df):
    if df.empty:
        return {'note': 'no audit results yet'}
    return {
        'risk_severity':    df['risk_severity'].value_counts(dropna=False).to_dict()    if 'risk_severity'    in df.columns else {},
        'hallucination_type': df['hallucination_type'].value_counts(dropna=False).to_dict() if 'hallucination_type' in df.columns else {},
        'safe_behavior':    df['safe_behavior'].value_counts(dropna=False).to_dict()    if 'safe_behavior'    in df.columns else {},
    }

report_summary = {
    'judge_trackA': judge_summary(judge_a),
    'judge_trackB': judge_summary(judge_b),
    'audit_trackA': audit_summary(audit_a),
    'audit_trackB': audit_summary(audit_b),
}
Path('outputs/report_summary.json').write_text(
    json.dumps(report_summary, indent=2), encoding='utf-8'
)
print(json.dumps(report_summary, indent=2))

## 5. Final Report Notes

Use the generated CSVs and JSON files to write the final narrative.
Replace each bullet with your actual numbers from this run.

**Does reasoning improve QA?**  
Compare `exact_match` and `rouge_l` for Track A vs Track B.

**Does reasoning increase cost or latency?**  
Compare `mean_output_tokens` and `mean_generation_time_s`.

**Does reasoning increase hallucinations?**  
Compare judge error counts and `hallucination_type` distributions in audit.

**Should reasoning be hidden or shown?**  
Discuss `reasoning_clarity` × `safe_behavior` from the manual audit.

**How unsafe is the model in edge cases?**  
Report high-risk row counts and `dangerous_advice` rate from audit.

**Include six worked examples** (3 good, 3 bad) drawn from
`outputs/trackA/predictions.csv` — one per row, with your written remarks.

---
*State clearly that this is a learning artefact, not a clinical product.*